# Phase 4 — Comparaison de modèles et évaluation (FD001)

> **En 30 secondes**
> - **Question** : XGBoost (Phase 3) est-il justifié face à d'autres modèles, et où placer une pénalité des surestimations (le cas dangereux) pour rendre un modèle plus prudent ?
> - **Méthode** : quatre modèles à réglages de base sur les mêmes 5 plis de validation croisée, tests appariés moteur par moteur ; une grille de pénalités placées dans l'apprentissage ; le test officiel évalué une seule fois.
> - **Résultat principal** (`docs/resultats/05_comparaison_cv.csv`) : RMSE CV de 16,96 (Random Forest), 16,87 (XGBoost Phase 3) et 16,85 (LightGBM) : égalité statistique (p > 0,5). XGBoost se justifie par sa vitesse (face à la Random Forest) et sa perte sur mesure, pas par sa précision.
> - **Pénalité** : changer 13/10 dans la note ne change aucune prédiction ; dans l'apprentissage (α de 1 à 3), XGBoost passe de 2 386 à 1 590 lignes surestimées de plus de 20 cycles (−33 %), au prix d'un RMSE CV de 17,68 au lieu de 16,86 et de 64 % de fausses alertes en plus.
> - **Limite principale** : sur le test officiel (`docs/resultats/05_comparaison_test.csv`, XGBoost Phase 3 : RMSE 19,70, score NASA 852,9), ni la Random Forest ni LightGBM ne sont significativement différents de XGBoost Phase 3 : 100 moteurs ne suffisent pas à les départager.

**Sommaire**

- [A. Préparation et règles](#partie-a)
- [B. Comparaison à réglages de base](#partie-b)
- [C. Où mettre la pénalité des surestimations ?](#partie-c)
- [D. Évaluation sur le test officiel](#partie-d)
- [E. Les moteurs surestimés de plus de 20 cycles](#partie-e)
- [F. Synthèse et fichiers de résultats](#partie-f)
- [G. Démo : tester un moteur que les modèles n'ont jamais vu](#partie-g)

**Exécution** : quelques minutes de bout en bout (durées affichées à la fin des sections F et G). Les chiffres de l'encadré sont vérifiés par le code en F.4.

<a id="partie-a"></a>
## A. Préparation et règles

### A.1 Quatre règles fixées avant tout résultat

1. **Le test officiel ne sert jamais à choisir** (ni features, ni modèle, ni réglage, ni seuil) : sinon son score deviendrait optimiste. Ici, il sert seulement à mesurer les modèles de la section B et à la démonstration de la section C.1.
2. **Les mêmes 5 plis pour tous les modèles** (`GroupKFold` par moteur, graine 42, comme en Phase 3) : un écart entre deux modèles ne peut donc pas venir du découpage.
3. **Le score NASA d'évaluation garde ses constantes officielles 13 / 10** : c'est la note, on n'y touche pas.
4. **La prudence se règle dans l'apprentissage** : pour réduire les surestimations, on change la perte que le modèle minimise (section C), jamais la note.

Un résultat non significatif ou mixte est écrit comme tel.

### A.2 Imports et réglages

**Ce qu'on fait** : on importe les fonctions du dossier `src/` (testées par pytest) au lieu de recopier le pipeline ici, et on fixe les seuils utilisés partout.
**Les modules** : `features.py` prépare les données (Phases 1 à 3), `evaluation.py` calcule les métriques, la validation croisée et les tests appariés, `config_modele.py` fabrique un modèle à partir d'un petit dictionnaire de réglages.

In [ ]:
import sys
sys.path.insert(0, "..")  # le dossier parent de notebooks/ (racine du depot) devient importable : "from src ..." marche

import warnings
warnings.filterwarnings("ignore", message="IProgress not found")  # masque un avertissement d'affichage, sans effet sur les calculs

# bibliotheques standard de Python
import copy  # deepcopy : copie independante d'un dictionnaire (l'original reste intact)
import os    # chemins de fichiers et dossiers
import time  # chronometres

# calcul, tableaux, figures, tests statistiques
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats  # correlation de Spearman, test de Mann-Whitney

# fonctions du projet (dossier src/, testees par pytest)
from src.config_modele import CONFIG_PHASE3, construire_fabrique_depuis_config
from src.evaluation import (
    calculer_plis,                          # les 5 plis de validation croisee par moteur
    comparer_apparie,                       # test t apparie + test de Wilcoxon signe
    demontrer_constantes_score_degenerees,  # demonstration de la section C.1
    erreurs_par_moteur,                     # RMSE moteur par moteur
    evaluer_en_validation_croisee,          # entraine et evalue un modele sur les 5 plis
    evaluer_sur_test,                       # evalue un modele deja entraine sur le test officiel
    nb_surestimes,                          # nombre de predictions surestimees de plus d'un seuil
    rmse,                                   # racine de l'erreur quadratique moyenne
    score_nasa,                             # score asymetrique NASA (13 / 10)
)
from src.features import preparer_test, preparer_train

SEUIL_SURESTIMATION = 20  # cycles : une surestimation de plus de 20 cycles est comptee comme dangereuse (choix Phase 3)
SEUIL_ALERTE = 30         # cycles : RUL predite <= 30 -> envoi en maintenance ; HYPOTHESE DE TRAVAIL, pas une valeur officielle
PLAFOND_RUL = 125         # plafond de la RUL (Phase 2) : la cible d'entrainement ne depasse jamais 125

DOSSIER_FIGURES = os.path.join("..", "docs", "figures")  # figures versionnees du projet
DOSSIER_RESULTATS = os.path.join("..", "docs", "resultats")  # fichiers CSV de resultats (section F)
os.makedirs(DOSSIER_FIGURES, exist_ok=True)  # cree le dossier s'il manque (aucune erreur s'il existe deja)
os.makedirs(DOSSIER_RESULTATS, exist_ok=True)

# couleurs des figures : 3 teintes bien distinctes, toujours la meme teinte pour la meme chose d'une figure
# a l'autre ; le texte et les reperes restent en gris fonce ou en noir
COULEUR_1 = "#2a78d6"  # bleu
COULEUR_2 = "#eb6834"  # orange
COULEUR_3 = "#1baf7a"  # vert d'eau
COULEUR_TEXTE = "#52514e"  # gris fonce (reperes, annotations)
plt.rcParams["axes.grid"] = True  # quadrillage sur toutes les figures
plt.rcParams["axes.axisbelow"] = True  # quadrillage dessine sous les barres et les points
plt.rcParams["grid.color"] = "#e4e3df"  # quadrillage gris clair, discret
plt.rcParams["axes.spines.top"] = False  # pas de bord en haut des graphiques
plt.rcParams["axes.spines.right"] = False  # pas de bord a droite

pd.set_option("display.width", 220)  # tableaux affiches sur une ligne plus large
pd.set_option("display.max_columns", 40)  # toutes les colonnes affichees
debut_notebook = time.perf_counter()  # chronometre : depart du calcul des sections A a F

### A.3 Données d'entraînement et de test

**Ce qu'on fait** : `preparer_train` refait le pipeline des Phases 1 à 3 sur `train_FD001.txt` (capteurs constants retirés, mise à l'échelle min-max, lissage, RUL plafonnée à 125, features glissantes) ; `preparer_test` applique au test les **mêmes** réglages, appris sur le train, sans rien réapprendre.
**Comment lire le résultat** : 20 631 lignes (une par cycle) et 46 features côté entraînement ; 100 lignes côté test, une par moteur au dernier cycle observé, à comparer à la RUL vraie de `RUL_FD001.txt`.

In [ ]:
contexte_train = preparer_train(fenetre=5, plafond=PLAFOND_RUL)  # pipeline des Phases 1 a 3 sur le train (statistiques sur 5 cycles)
contexte_test = preparer_test(contexte_train)  # memes transformations sur le test, avec les reglages appris sur le train

X = contexte_train["X"]            # les 46 features, une ligne par cycle
y = contexte_train["y"]            # la cible : RUL plafonnee a 125
groups = contexte_train["groups"]  # numero de moteur de chaque ligne (sert a GroupKFold)

X_test_dernier = contexte_test["X_test_dernier"]  # features du dernier cycle observe de chaque moteur test
y_test_vraie = contexte_test["y_test_vraie"]      # RUL vraie BRUTE de RUL_FD001.txt (non plafonnee)

print(f"Train : {X.shape[0]} lignes, {X.shape[1]} features, {groups.nunique()} moteurs")
print(f"Test  : {X_test_dernier.shape[0]} moteurs (un point par moteur, au dernier cycle observe)")
assert X.shape == (20631, 46)  # garde-fou : memes dimensions qu'en Phase 3
assert X_test_dernier.shape == (100, 46)  # 100 moteurs test, memes 46 features

### A.4 Les 5 plis de validation croisée, calculés une seule fois

**Ce qu'on fait** : on répartit les 100 moteurs d'entraînement en 5 groupes de 20 ; chaque groupe sert une fois de validation pendant que les 4 autres servent à apprendre.
**Pourquoi par moteur** : un moteur entier est soit en apprentissage, soit en validation ; sinon le modèle verrait des cycles voisins du même moteur des deux côtés et paraîtrait meilleur qu'il n'est.
**Comment lire le résultat** : 20 moteurs en validation dans chaque pli ; `calculer_plis` vérifie aussi qu'aucun moteur n'est des deux côtés. Ces plis servent à tous les modèles du notebook.

In [ ]:
plis = calculer_plis(X, y, groups, n_splits=5, random_state=42)  # 5 plis par moteur, melange de graine 42 (comme en Phase 3)

NB_MOTEURS_TRAIN = int(groups.nunique())  # 100 moteurs d'entrainement
for numero_pli, (idx_train, idx_val) in enumerate(plis):  # idx_train, idx_val : positions des lignes de chaque cote
    nb_moteurs_val = groups.iloc[idx_val].nunique()  # nombre de moteurs mis de cote pour la validation dans ce pli
    print(f"Pli {numero_pli + 1} : {len(idx_train)} lignes d'entrainement, {len(idx_val)} lignes de validation "
          f"({nb_moteurs_val} moteurs)")

### A.5 Les colonnes des tableaux

`resumer_cv` transforme le résultat d'une validation croisée en une ligne de tableau, toujours avec les mêmes colonnes :

| Colonne | Sens |
|---|---|
| `rmse_cv_moyen`, `rmse_cv_ecart_type` | erreur (en cycles) moyenne sur les 5 plis, et sa variation d'un pli à l'autre |
| `mae_cv_moyen` | erreur absolue moyenne, moins sensible aux grosses erreurs isolées |
| `nasa_cv_moyen` | moyenne des scores NASA (13/10) des 5 plis ; environ 4 000 lignes par pli, d'où des valeurs proches de 30 000 (le score du test ne porte que sur 100 points) |
| `surestimes_oof` | lignes surestimées de plus de 20 cycles parmi les 20 631 prédictions hors-pli (OOF : chaque ligne est prédite par le modèle du pli où elle était en validation) |
| `alertes_manquees_oof` | RUL vraie ≤ 30 mais prédiction > 30 : le moteur aurait dû partir en maintenance (dangereux) |
| `fausses_alertes_oof` | RUL vraie > 30 mais prédiction ≤ 30 : maintenance inutile (coûteux, pas dangereux) |
| `biais_oof` | moyenne de (prédiction − vérité) ; négatif = le modèle sous-estime en moyenne (prudent) |
| `temps_entrainement_5_plis_s`, `prediction_ms_par_moteur` | entraînement des 5 plis (s) ; prédiction de toute la trajectoire d'un moteur (ms) |

`resumer_test` fait de même pour le test. `metriques_par_moteur` calcule RMSE, pénalité NASA et surestimations **moteur par moteur**, pour les tests appariés.

In [ ]:
def resumer_cv(nom, resultats, nb_moteurs):
    """
    Resume les resultats d'une validation croisee en une ligne de tableau (un dictionnaire).

    Parameters
    ----------
    nom : str
        Le nom du modele (premiere colonne du tableau).
    resultats : dict
        Le dictionnaire renvoye par `evaluer_en_validation_croisee`.
    nb_moteurs : int
        Nombre de moteurs predits au total (100), pour le temps de prediction par moteur.

    Returns
    -------
    dict
        Les colonnes communes a tous les tableaux de validation croisee de ce notebook.
    """
    # on recopie les metriques utiles, une cle a la fois (sens de chaque colonne : section A.5)
    ligne = {}
    ligne["modele"] = nom
    ligne["rmse_cv_moyen"] = resultats["rmse_moyen"]
    ligne["rmse_cv_ecart_type"] = resultats["rmse_ecart_type"]
    ligne["mae_cv_moyen"] = resultats["mae_moyen"]
    ligne["nasa_cv_moyen"] = resultats["nasa_moyen"]
    ligne["surestimes_oof"] = resultats["surestimes_oof"]
    ligne["alertes_manquees_oof"] = resultats["alertes_manquees_oof"]
    ligne["fausses_alertes_oof"] = resultats["fausses_alertes_oof"]
    ligne["biais_oof"] = resultats["biais_oof"]
    ligne["temps_entrainement_5_plis_s"] = resultats["temps_entrainement_total"]
    # temps de prediction total des 5 plis (secondes) divise par le nombre de moteurs, converti en millisecondes
    ligne["prediction_ms_par_moteur"] = 1000.0 * resultats["temps_prediction_total"] / nb_moteurs
    return ligne


def resumer_test(nom, resultats_test):
    """
    Resume l'evaluation d'un modele sur le test officiel en une ligne de tableau (un dictionnaire).

    Parameters
    ----------
    nom : str
        Le nom du modele.
    resultats_test : dict
        Le dictionnaire renvoye par `evaluer_sur_test`.

    Returns
    -------
    dict
        RMSE (contre la RUL brute et contre la RUL plafonnee), MAE, score NASA, surestimes > 20,
        alertes manquees et fausses alertes, sur les 100 moteurs test.
    """
    # memes metriques que resumer_cv, mais sur les 100 moteurs test (un point par moteur)
    ligne = {}
    ligne["modele"] = nom
    ligne["rmse_test"] = resultats_test["rmse"]
    ligne["rmse_test_rul_plafonnee"] = resultats_test["rmse_contre_rul_plafonnee"]
    ligne["mae_test"] = resultats_test["mae"]
    ligne["nasa_test"] = resultats_test["score_nasa"]
    ligne["surestimes_test"] = resultats_test["nb_surestimes"]
    ligne["alertes_manquees_test"] = resultats_test["alertes_manquees"]
    ligne["fausses_alertes_test"] = resultats_test["fausses_alertes"]
    return ligne


def metriques_par_moteur(y_true, y_pred, groups):
    """
    Calcule trois metriques moteur par moteur : RMSE, penalite NASA (13/10) et surestimations > 20.

    Parameters
    ----------
    y_true : array-like
        Les vraies valeurs (une par ligne).
    y_pred : array-like
        Les predictions (une par ligne, meme ordre).
    groups : array-like
        Le numero de moteur de chaque ligne (sur le test : une seule ligne par moteur).

    Returns
    -------
    pandas.DataFrame
        Une ligne par moteur (triee par numero) : "unit_number", "rmse", "nasa" (somme des penalites
        NASA des lignes du moteur) et "surestimes" (lignes surestimees de plus de 20 cycles).
    """
    vrais = np.asarray(y_true, dtype=float)  # tableaux numpy : on pourra les filtrer avec un masque
    predits = np.asarray(y_pred, dtype=float)
    groupes = np.asarray(groups)

    lignes = []
    for moteur_id in np.unique(groupes):  # un moteur a la fois, dans l'ordre croissant des numeros
        masque = groupes == moteur_id  # Vrai sur les lignes de ce moteur
        ligne = {}
        ligne["unit_number"] = moteur_id
        ligne["rmse"] = rmse(vrais[masque], predits[masque])
        ligne["nasa"] = score_nasa(vrais[masque], predits[masque])
        ligne["surestimes"] = nb_surestimes(vrais[masque], predits[masque], seuil=SEUIL_SURESTIMATION)
        lignes.append(ligne)
    return pd.DataFrame(lignes)

<a id="partie-b"></a>
## B. Comparaison à réglages de base, sur les mêmes plis

**Ce qu'on fait** : on évalue quatre modèles avec des réglages par défaut (aucun n'est optimisé, pour comparer à armes égales). **Pourquoi** : vérifier si XGBoost, choisi en Phase 3, fait mieux que des alternatives.

| Modèle | Réglages |
|---|---|
| Référence naïve | prédit toujours la moyenne de la cible d'entraînement : la borne basse |
| Random Forest | 200 arbres, profondeur non bornée (défaut scikit-learn) |
| XGBoost Phase 3 | 200 arbres, profondeur 5, pas 0,05 (exactement la Phase 3) |
| LightGBM | 200 arbres, pas 0,05, 31 feuilles par arbre (défaut LightGBM), mode déterministe |

**Contrôle de reproduction** : XGBoost Phase 3 doit redonner un RMSE CV moyen de 16,87 et un score NASA CV moyen de 29 782,6 (le document de Phase 3 écrit 16,86 / 29 729,3, obtenus dans un autre environnement : voir sa note de reproductibilité).

In [ ]:
# chaque modele est decrit par un petit dictionnaire (format de src/config_modele.py) : famille, perte
# d'apprentissage, parametres de la perte, hyperparametres et nom lisible
CONFIG_REFERENCE_NAIVE = {
    "famille": "reference_naive", "perte": "native", "parametres_perte": {}, "hyperparametres": {},
    "nom": "Reference naive (moyenne)",
}
CONFIG_RANDOM_FOREST = {
    "famille": "random_forest", "perte": "native", "parametres_perte": {},
    "hyperparametres": {"n_estimators": 200, "random_state": 42, "n_jobs": -1},  # 200 arbres, graine 42, tous les coeurs
    "nom": "Random Forest (base)",
}
CONFIG_XGB_PHASE3 = copy.deepcopy(CONFIG_PHASE3)  # exactement la Phase 3 (copie : l'original reste intact)
CONFIG_LIGHTGBM_BASE = {
    "famille": "lightgbm", "perte": "native", "parametres_perte": {},
    "hyperparametres": {"n_jobs": 4},  # 4 coeurs ; le reste : valeurs de fabrique_lightgbm (200 arbres, pas 0.05, ...)
    "nom": "LightGBM (base)",
}

# les 4 modeles compares, dans l'ordre des tableaux
CONFIGS_BASE = {}
CONFIGS_BASE["Reference naive"] = CONFIG_REFERENCE_NAIVE
CONFIGS_BASE["Random Forest"] = CONFIG_RANDOM_FOREST
CONFIGS_BASE["XGBoost Phase 3"] = CONFIG_XGB_PHASE3
CONFIGS_BASE["LightGBM"] = CONFIG_LIGHTGBM_BASE

resultats_cv_base = {}  # nom du modele -> dictionnaire renvoye par evaluer_en_validation_croisee
for nom_modele in CONFIGS_BASE:  # un modele a la fois, toujours sur les memes plis
    fabrique = construire_fabrique_depuis_config(CONFIGS_BASE[nom_modele])  # fonction qui renvoie un modele neuf a chaque appel
    debut = time.perf_counter()  # chronometre de ce modele
    resultats_cv_base[nom_modele] = evaluer_en_validation_croisee(  # 5 entrainements, un par pli
        fabrique, X, y, groups, plis, seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE
    )
    duree = time.perf_counter() - debut
    print(f"{nom_modele:16s} : RMSE CV moyen = {resultats_cv_base[nom_modele]['rmse_moyen']:.2f} ({duree:.1f} s)")

# controle de reproduction de la Phase 3 dans cet environnement : le notebook s'arrete si les chiffres changent
assert round(resultats_cv_base["XGBoost Phase 3"]["rmse_moyen"], 2) == 16.87
assert round(resultats_cv_base["XGBoost Phase 3"]["nasa_moyen"], 1) == 29782.6

### B.1 Tableau comparatif (validation croisée)

**Comment lire** : une ligne par modèle, colonnes expliquées en A.5 ; plus petit = mieux, sauf `biais_oof` (le plus proche de 0 = sans biais). Les colonnes de temps changent d'une exécution à l'autre, les autres non.

In [ ]:
lignes = []  # une ligne de tableau par modele
for nom_modele in resultats_cv_base:
    lignes.append(resumer_cv(nom_modele, resultats_cv_base[nom_modele], NB_MOTEURS_TRAIN))
tableau_cv_base = pd.DataFrame(lignes).set_index("modele")  # le nom du modele sert d'etiquette de ligne
tableau_cv_base.round(2)  # affichage arrondi a 2 decimales (les valeurs exactes restent dans tableau_cv_base)

### B.2 Les écarts sont-ils significatifs ? Tests appariés par moteur

**Notion — test apparié** : on compare deux modèles **moteur par moteur** (les mêmes 100 moteurs), pas seulement leurs moyennes. Exemple : sur un moteur difficile, RMSE 25 (Random Forest) et 24 (XGBoost) ; sur un moteur facile, 10 et 11. Les RMSE vont de 10 à 25, mais les différences (+1, −1) restent petites : c'est sur elles que porte le test, la difficulté propre à chaque moteur s'annule.
- **Test t apparié** : la différence moyenne (modèle − XGBoost) est-elle nulle ? **Wilcoxon signé** : même question avec les rangs, sans supposer de loi normale.
- **p-valeur** : probabilité d'observer un écart au moins aussi grand si les deux modèles étaient en réalité équivalents. Petite p-valeur = écart difficile à attribuer au hasard.
- **Bonferroni** : 3 comparaisons, donc on exige p < 0,05 / 3 ≈ 0,017 pour chacune (risque global de fausse découverte gardé à 5 %).

**Comment lire** : différence moyenne négative = le modèle a un RMSE par moteur plus petit que XGBoost.

In [ ]:
# RMSE hors-pli de chaque moteur, pour chaque modele (100 lignes, triees par numero de moteur)
erreurs_moteur_base = {}
for nom_modele in resultats_cv_base:
    erreurs_moteur_base[nom_modele] = erreurs_par_moteur(y, resultats_cv_base[nom_modele]["predictions_oof"], groups)

rmse_moteur_xgb = erreurs_moteur_base["XGBoost Phase 3"]["rmse"].to_numpy()  # reference des comparaisons
SEUIL_BONFERRONI = 0.05 / 3  # 3 comparaisons contre XGBoost Phase 3 : seuil de 0,017 pour chacune

lignes = []
for nom_modele in resultats_cv_base:
    if nom_modele == "XGBoost Phase 3":
        continue  # on ne compare pas XGBoost a lui-meme
    rmse_moteur_modele = erreurs_moteur_base[nom_modele]["rmse"].to_numpy()  # 100 RMSE, memes moteurs, meme ordre
    comparaison = comparer_apparie(rmse_moteur_modele, rmse_moteur_xgb)  # tests sur les 100 differences modele - XGBoost
    ligne = {}
    ligne["modele (vs XGBoost Phase 3)"] = nom_modele
    ligne["n_moteurs"] = comparaison["n"]  # nombre de paires comparees
    ligne["difference_moyenne_rmse_par_moteur"] = comparaison["difference_moyenne"]  # negatif = modele meilleur
    ligne["moteurs_ou_le_modele_est_meilleur"] = int(np.sum(rmse_moteur_modele < rmse_moteur_xgb))  # compte les Vrai
    ligne["p_valeur_t"] = comparaison["p_valeur_t"]
    ligne["p_valeur_wilcoxon"] = comparaison["p_valeur_wilcoxon"]
    # significatif seulement si les DEUX tests passent sous le seuil de Bonferroni
    ligne["significatif_bonferroni"] = (comparaison["p_valeur_t"] < SEUIL_BONFERRONI) and (
        comparaison["p_valeur_wilcoxon"] < SEUIL_BONFERRONI)
    lignes.append(ligne)
tableau_tests_base = pd.DataFrame(lignes).set_index("modele (vs XGBoost Phase 3)")
tableau_tests_base.round(4)  # 4 decimales : 0.0000 veut dire p < 0,00005

### B.3 Figure de comparaison

**Comment lire** : un critère par panneau (jamais deux échelles sur le même axe). Panneau RMSE : la barre est la moyenne des 5 plis, les points sont les 5 plis. La référence naïve, hors échelle, est citée dans le titre.

In [ ]:
noms_compares = ["Random Forest", "XGBoost Phase 3", "LightGBM"]  # la reference naive est hors echelle : pas dessinee
positions = np.arange(len(noms_compares))  # positions 0, 1, 2 des barres sur l'axe horizontal

fig, axes = plt.subplots(2, 2, figsize=(12, 8))  # 4 panneaux : 2 lignes x 2 colonnes

# (1) RMSE CV : moyenne (barre) et valeur de chaque pli (points)
ax = axes[0, 0]
for position, nom_modele in zip(positions, noms_compares):  # zip : une position et un nom a chaque tour
    resultats = resultats_cv_base[nom_modele]
    ax.bar(position, resultats["rmse_moyen"], width=0.6, color=COULEUR_1, alpha=0.85)
    ax.scatter(np.full(5, position), resultats["rmse_par_pli"], color="black", s=18, zorder=3)  # 5 points a la meme abscisse
    ax.text(position, 19.5, f"moyenne {resultats['rmse_moyen']:.2f}", ha="center", fontsize=9)  # au-dessus des points
ax.set_ylim(14, 20)
ax.set_ylabel("RMSE (cycles)")
ax.set_title("RMSE CV moyen (barre) et par pli (points)")

# (2) score NASA CV moyen
ax = axes[0, 1]
for position, nom_modele in zip(positions, noms_compares):
    valeur = resultats_cv_base[nom_modele]["nasa_moyen"]
    ax.bar(position, valeur, width=0.6, color=COULEUR_1, alpha=0.85)
    ax.text(position, valeur * 1.01, f"{valeur:,.0f}".replace(",", " "), ha="center", fontsize=9)  # milliers separes par une espace
ax.set_ylabel("score NASA (13/10), moyenne des 5 plis")
ax.set_title("Score NASA CV moyen (plus bas = mieux)")

# (3) surestimations dangereuses : lignes OOF surestimees de plus de 20 cycles
ax = axes[1, 0]
for position, nom_modele in zip(positions, noms_compares):
    valeur = resultats_cv_base[nom_modele]["surestimes_oof"]
    ax.bar(position, valeur, width=0.6, color=COULEUR_2, alpha=0.85)
    ax.text(position, valeur * 1.01, str(valeur), ha="center", fontsize=9)
ax.set_ylabel("lignes OOF (sur 20 631)")
ax.set_title("Surestimés de plus de 20 cycles (OOF)")

# (4) temps d'entrainement cumule sur les 5 plis
ax = axes[1, 1]
for position, nom_modele in zip(positions, noms_compares):
    valeur = resultats_cv_base[nom_modele]["temps_entrainement_total"]
    ax.bar(position, valeur, width=0.6, color=COULEUR_TEXTE, alpha=0.85)
    ax.text(position, valeur * 1.02, f"{valeur:.1f} s", ha="center", fontsize=9)
ax.set_ylabel("secondes (5 plis)")
ax.set_title("Temps d'entraînement (5 plis)")

for ax in axes.ravel():  # ravel : les 4 panneaux a la suite ; memes etiquettes de modeles sous chacun
    ax.set_xticks(positions)
    ax.set_xticklabels(noms_compares)

rmse_naive = resultats_cv_base["Reference naive"]["rmse_moyen"]
fig.suptitle(f"Phase 4 — comparaison à réglages de base, mêmes 5 plis GroupKFold "
             f"(référence naïve hors échelle : RMSE CV = {rmse_naive:.1f})")
plt.tight_layout()  # evite que les titres et etiquettes se chevauchent
plt.savefig(os.path.join(DOSSIER_FIGURES, "05_comparaison_modeles.png"), dpi=150)  # figure versionnee
plt.show()

### B.4 Conclusion de la section B

- **Les modèles apprennent** : RMSE CV de 41,7 pour la référence naïve contre environ 16,9 pour les trois ensembles d'arbres (p < 0,0001).
- **Précision : égalité statistique** entre Random Forest (16,96), XGBoost Phase 3 (16,87) et LightGBM (16,85) : 0,11 cycle d'écart au plus, contre un écart-type d'environ 1,3 d'un pli à l'autre ; tests appariés non significatifs (p = 0,78 / 0,58 et 0,79 / 0,80).
- **Prudence : pas de différence nette non plus** (surestimés > 20 : 2 302 / 2 384 / 2 316 ; alertes manquées : 565 / 552 / 568).
- **Temps** : XGBoost et LightGBM s'entraînent en quelques secondes pour les 5 plis, la Random Forest plusieurs fois plus lentement (les secondes exactes varient d'une exécution à l'autre).

**Réponse à « XGBoost est-il justifié ? »** : pas par sa précision (égalité), mais parce qu'il s'entraîne vite et accepte une perte d'apprentissage sur mesure (section C), ce que la Random Forest de scikit-learn ne permet pas. LightGBM a les mêmes atouts : il est gardé dans la grille de la section C.

<a id="partie-c"></a>
## C. Rendre le modèle plus prudent : où mettre la pénalité ?

**Rappel — score NASA** : pour un écart d = prédiction − vérité, la pénalité vaut exp(d / 10) − 1 si le modèle surestime (d ≥ 0) et exp(−d / 13) − 1 s'il sous-estime ; le score est la somme des pénalités. Exemple : surestimer de 20 cycles coûte e² − 1 ≈ 6,4 ; sous-estimer de 20 cycles coûte e^(20/13) − 1 ≈ 3,7.

### C.1 Fausse piste : changer 13 et 10 dans la note ne change aucune prédiction

**Ce qu'on fait** : on multiplie les deux constantes par un facteur k et on recalcule le score des **mêmes** prédictions test de XGBoost Phase 3. **Pourquoi** : l'idée de départ était de « faire varier 10 et 13 » pour rendre le modèle plus prudent ; on montre que cela change la note, pas le modèle. Le test sert ici à une démonstration, pas à un choix.

In [ ]:
# modele Phase 3 entraine sur les 100 moteurs d'entrainement (comme en Phase 3), puis predictions des 100 moteurs test
modele_phase3_demo = construire_fabrique_depuis_config(CONFIG_XGB_PHASE3)()  # la fabrique, appelee avec () : un modele neuf
modele_phase3_demo.fit(X, y)
predictions_test_phase3_demo = np.clip(modele_phase3_demo.predict(X_test_dernier), a_min=0, a_max=None)  # RUL negative -> 0

# les memes predictions, notees avec les constantes 13 x k et 10 x k
tableau_constantes = demontrer_constantes_score_degenerees(y_test_vraie, predictions_test_phase3_demo,
                                                          facteurs=(1, 2, 5, 10, 100, 1000))
tableau_constantes["rmse_test"] = rmse(y_test_vraie, predictions_test_phase3_demo)  # predictions inchangees : meme RMSE sur chaque ligne
tableau_constantes.round(3)

**Comment lire** : les prédictions et le RMSE sont identiques sur toutes les lignes, seule la note change ; plus k est grand, plus le score tend vers 0, et ce serait vrai pour n'importe quel modèle, même mauvais. « Optimiser » k reviendrait à choisir la note qui flatte le plus.

### C.2 La bonne piste : la pénalité dans la perte d'apprentissage

**Notion — perte d'apprentissage** : la quantité que le modèle rend la plus petite possible pendant l'entraînement. La changer change les prédictions ; la note d'évaluation reste le score NASA 13/10. Deux pertes sur mesure existent dans `src/modeles.py` (testées par pytest) :
- **quadratique asymétrique** : erreur² × α si le modèle surestime, erreur² sinon. Exemple avec α = 2 : surestimer de 10 cycles coûte 200, sous-estimer de 10 cycles coûte 100 ;
- **forme NASA** : la forme du score NASA, avec des constantes h_sous et h_sur réglables (prolongée par une parabole au-delà de 50 cycles d'écart, pour éviter des exponentielles géantes).

**Ce qu'on fait** : une expérience sur une petite grille, mêmes plis : XGBoost (réglages Phase 3) et LightGBM (réglages de base) avec α ∈ {1 ; 1,5 ; 2 ; 3} ; XGBoost avec la perte NASA (13, 10) et (13, 5), à 200 arbres puis à 1 000 et 2 000 arbres au pas 0,1. Aucun modèle n'est choisi ici.
**Pourquoi 1 000 et 2 000 arbres** : avec la perte NASA, chaque arbre ne déplace les prédictions que d'environ « pas × h » cycles (0,05 × 5 = 0,25 cycle pour h = 5) ; 200 arbres peuvent ne pas suffire à finir l'apprentissage.

In [ ]:
reglages_penalite = []  # une entree par reglage : nom lisible, groupe (pour la figure) et configuration

for alpha in [1.0, 1.5, 2.0, 3.0]:  # XGBoost, perte quadratique asymetrique, reglages Phase 3
    reglage = {}
    reglage["reglage"] = f"XGBoost quadratique alpha={alpha:g}"  # :g affiche 1.0 comme "1" et 1.5 comme "1.5"
    reglage["groupe"] = "XGBoost quadratique asymétrique"
    reglage["config"] = {"famille": "xgboost", "perte": "quadratique_asymetrique",
                         "parametres_perte": {"alpha": alpha}, "hyperparametres": {"n_jobs": 4}}
    reglages_penalite.append(reglage)

for alpha in [1.0, 1.5, 2.0, 3.0]:  # LightGBM, perte quadratique asymetrique, reglages de base
    reglage = {}
    reglage["reglage"] = f"LightGBM quadratique alpha={alpha:g}"
    reglage["groupe"] = "LightGBM quadratique asymétrique"
    reglage["config"] = {"famille": "lightgbm", "perte": "quadratique_asymetrique",
                         "parametres_perte": {"alpha": alpha}, "hyperparametres": {"n_jobs": 4}}
    reglages_penalite.append(reglage)

for h_sous, h_sur in [(13.0, 10.0), (13.0, 5.0)]:  # XGBoost, perte de forme NASA (h_sur = 5 : plus prudent)
    for n_arbres, pas in [(200, 0.05), (1000, 0.1), (2000, 0.1)]:  # reglages Phase 3, puis beaucoup plus d'arbres
        reglage = {}
        reglage["reglage"] = f"XGBoost NASA ({h_sous:g},{h_sur:g}) {n_arbres} arbres pas {pas:g}"
        if n_arbres == 200:
            reglage["groupe"] = "XGBoost perte NASA, 200 arbres"
        else:
            reglage["groupe"] = "XGBoost perte NASA, 1000-2000 arbres"
        reglage["config"] = {"famille": "xgboost", "perte": "nasa",
                             "parametres_perte": {"h_sous": h_sous, "h_sur": h_sur},
                             "hyperparametres": {"n_estimators": n_arbres, "learning_rate": pas, "n_jobs": 4}}
        reglages_penalite.append(reglage)

resultats_cv_penalite = {}  # nom du reglage -> resultats de la validation croisee (memes 5 plis)
for reglage in reglages_penalite:
    fabrique = construire_fabrique_depuis_config(reglage["config"])
    debut = time.perf_counter()
    resultats_cv_penalite[reglage["reglage"]] = evaluer_en_validation_croisee(
        fabrique, X, y, groups, plis, seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE
    )
    print(f"{reglage['reglage']:45s} ({time.perf_counter() - debut:5.1f} s)")  # nom, puis duree des 5 plis

**Tableau de la grille** (la première ligne est la référence de la section B). **Comment lire** : quand α augmente, `surestimes_oof` doit baisser ; `rmse_cv_moyen`, `fausses_alertes_oof` et `biais_oof` montrent le prix payé.

In [ ]:
lignes = []
# premiere ligne : XGBoost Phase 3 de la section B, comme reference
ligne_reference = resumer_cv("XGBoost Phase 3 (perte native)", resultats_cv_base["XGBoost Phase 3"], NB_MOTEURS_TRAIN)
ligne_reference["groupe"] = "référence"
lignes.append(ligne_reference)
for reglage in reglages_penalite:  # puis une ligne par reglage de la grille
    ligne = resumer_cv(reglage["reglage"], resultats_cv_penalite[reglage["reglage"]], NB_MOTEURS_TRAIN)
    ligne["groupe"] = reglage["groupe"]  # sert a la figure C.3
    lignes.append(ligne)
tableau_penalite = pd.DataFrame(lignes).set_index("modele")

# colonnes montrees (sans MAE ni temps de prediction, pour garder le tableau lisible)
colonnes_affichees = ["rmse_cv_moyen", "rmse_cv_ecart_type", "nasa_cv_moyen", "surestimes_oof",
                      "alertes_manquees_oof", "fausses_alertes_oof", "biais_oof", "temps_entrainement_5_plis_s"]
tableau_penalite[colonnes_affichees].round(2)

**Diagnostic de convergence de la perte NASA** : si les chiffres bougent peu entre 1 000 et 2 000 arbres, l'apprentissage a convergé ; s'ils bougent beaucoup entre 200 et 1 000 arbres, les 200 arbres de la Phase 3 n'avaient pas fini d'apprendre.

In [ ]:
for h_sous, h_sur in [(13, 10), (13, 5)]:  # les deux pertes NASA de la grille
    print(f"Perte NASA ({h_sous},{h_sur}) :")
    for n_arbres, pas in [(200, 0.05), (1000, 0.1), (2000, 0.1)]:
        nom_reglage = f"XGBoost NASA ({h_sous:g},{h_sur:g}) {n_arbres} arbres pas {pas:g}"  # meme nom qu'en C.2
        resultats = resultats_cv_penalite[nom_reglage]
        print(f"  {n_arbres:5d} arbres, pas {pas:4.2f} : RMSE CV = {resultats['rmse_moyen']:.2f}, "
              f"NASA CV = {resultats['nasa_moyen']:8.1f}, surestimes OOF > 20 = {resultats['surestimes_oof']:4d}, "
              f"biais OOF = {resultats['biais_oof']:+.2f}")

### C.3 Figure : le compromis précision / prudence

**Comment lire** : un point par réglage. Horizontalement le RMSE CV (à gauche = plus précis) ; verticalement les surestimés OOF > 20 (panneau de gauche) ou le score NASA CV (panneau de droite), en bas = plus prudent. Les points d'une même famille sont reliés par α croissant ; l'étoile noire est XGBoost Phase 3.

In [ ]:
# une ligne par groupe : (nom, couleur, marqueur, style de ligne, decalage de l'etiquette en points, alignement)
# les etiquettes XGBoost vont a gauche des points, celles de LightGBM a droite, pour ne pas se chevaucher
groupes_figure = [
    ("XGBoost quadratique asymétrique", COULEUR_1, "o", "-", (-7, -11), "right"),
    ("LightGBM quadratique asymétrique", COULEUR_2, "s", "-", (7, 3), "left"),
    ("XGBoost perte NASA, 200 arbres", COULEUR_3, "D", "none", (7, -12), "left"),
    ("XGBoost perte NASA, 1000-2000 arbres", COULEUR_3, "^", "none", (7, -10), "left"),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, colonne_y, titre_y in [(axes[0], "surestimes_oof", "surestimés > 20 cycles (lignes OOF)"),
                                (axes[1], "nasa_cv_moyen", "score NASA CV moyen (13/10)")]:  # un panneau a la fois
    for groupe, couleur, marqueur, style_ligne, decalage, alignement in groupes_figure:
        sous_tableau = tableau_penalite[tableau_penalite["groupe"] == groupe]  # les reglages de ce groupe
        ax.plot(sous_tableau["rmse_cv_moyen"], sous_tableau[colonne_y], marker=marqueur, linestyle=style_ligne,
                color=couleur, markersize=8, linewidth=2, label=groupe)
        for nom_reglage in sous_tableau.index:  # etiquette courte a cote de chaque point
            # on retire la partie du nom deja donnee par la legende
            etiquette = nom_reglage.replace("XGBoost quadratique ", "").replace("LightGBM quadratique ", "")
            etiquette = etiquette.replace("XGBoost NASA ", "").replace(" arbres pas ", " arb. pas ")
            decalage_point = decalage
            if "2000 arbres" in nom_reglage:  # 1000 et 2000 arbres sont tres proches : etiquette au-dessus
                decalage_point = (7, 4)
            ax.annotate(etiquette, (sous_tableau.loc[nom_reglage, "rmse_cv_moyen"], sous_tableau.loc[nom_reglage, colonne_y]),
                        textcoords="offset points", xytext=decalage_point, ha=alignement, fontsize=7, color=COULEUR_TEXTE)
    reference = tableau_penalite.loc["XGBoost Phase 3 (perte native)"]  # l'etoile noire
    ax.scatter(reference["rmse_cv_moyen"], reference[colonne_y], marker="*", s=250, color="black", zorder=4,
               label="XGBoost Phase 3 (perte native)")
    ax.set_xlim(16.0, 20.6)  # marge a gauche pour les etiquettes XGBoost
    ax.set_xlabel("RMSE CV moyen (cycles) — plus à gauche = plus précis")
    ax.set_ylabel(titre_y + " — plus bas = plus prudent")
axes[0].set_title("Précision vs surestimations dangereuses")
axes[1].set_title("Précision vs score NASA")
poignees, etiquettes = axes[0].get_legend_handles_labels()  # elements de legende du premier panneau
fig.legend(poignees, etiquettes, loc="lower center", ncol=5, fontsize=9, frameon=False)  # legende commune, sous les panneaux
fig.suptitle("Pénaliser la surestimation dans l'apprentissage : pas de gain gratuit (validation croisée, mêmes 5 plis)")
plt.tight_layout(rect=(0, 0.06, 1, 1))  # on laisse la place de la legende en bas
plt.savefig(os.path.join(DOSSIER_FIGURES, "05_penalite_surestimation.png"), dpi=150)
plt.show()

### C.4 Conclusion de la section C

1. **Changer 13 et 10 dans la note ne change rien au modèle** : RMSE test 19,70 pour tous les k, alors que le score passe de 852,9 (k = 1) à 0,13 (k = 1 000).
2. **La pénalité dans l'apprentissage fonctionne, avec un prix.** Quand α passe de 1 à 3, les surestimés OOF > 20 baissent de 2 386 à 1 590 (−33 %) pour XGBoost et de 2 316 à 1 675 (−28 %) pour LightGBM ; en échange (XGBoost), le RMSE CV monte de 16,86 à 17,68, les fausses alertes de 266 à 435 (+64 %) et le biais passe de −0,2 à −4,2 cycles.
3. **α = 1 redonne la Phase 3** (RMSE CV 16,86 contre 16,87) : la perte sur mesure est bien branchée.
4. **Perte NASA : attention à la convergence.** À 200 arbres, (13, 10) donne le meilleur compromis de la grille (RMSE 16,85 ; score NASA 22 235, soit −25 % ; surestimés 1 806, soit −24 %), mais ce modèle n'a pas fini d'apprendre (biais −2,7) : à 1 000 et 2 000 arbres, chiffres proches entre eux, il devient moins bon que la Phase 3 sur les deux critères (RMSE 17,51 ; score NASA 31 704). Le gain à 200 arbres vient donc de l'apprentissage inachevé, pas de la perte seule.
5. **Conséquence** : pénalité, nombre d'arbres et pas devraient être réglés **ensemble**. Ce notebook ne fait pas cette recherche et ne choisit aucun nouveau modèle : la recherche d'hyperparamètres de la Phase 4 est dans `notebooks/04d_hyperparametres.ipynb` (travail de Matisse, branche `main`).

<a id="partie-d"></a>
## D. Évaluation sur le test officiel (une seule fois)

**Ce qu'on fait** : les trois ensembles d'arbres de la section B sont entraînés sur les 100 moteurs d'entraînement, puis évalués sur le dernier cycle de chaque moteur test, contre la RUL **brute** de `RUL_FD001.txt` (prédictions négatives ramenées à 0). On ajoute le RMSE contre la RUL vraie plafonnée à 125, ce que le modèle pouvait réellement apprendre.
**Contrôle de reproduction** : XGBoost Phase 3 doit redonner RMSE 19,70, score NASA 852,9 et 9 surestimés.

In [ ]:
configs_finales = {}  # nom -> configuration des modeles evalues sur le test (ceux de la section B)
configs_finales["XGBoost Phase 3"] = CONFIG_XGB_PHASE3
configs_finales["Random Forest"] = CONFIG_RANDOM_FOREST
configs_finales["LightGBM"] = CONFIG_LIGHTGBM_BASE

moteurs_test_ordre = contexte_test["moteurs_test"]  # numeros des moteurs test, meme ordre que y_test_vraie
resultats_test = {}  # nom -> dictionnaire renvoye par evaluer_sur_test
for nom_modele in configs_finales:
    modele_final = construire_fabrique_depuis_config(configs_finales[nom_modele])()  # modele neuf
    modele_final.fit(X, y)  # entraine sur les 100 moteurs d'entrainement
    resultats_test[nom_modele] = evaluer_sur_test(modele_final, X_test_dernier, y_test_vraie, plafond=PLAFOND_RUL,
                                                  seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE)

# controle de reproduction du test de la Phase 3 (le notebook s'arrete si un chiffre change)
assert round(resultats_test["XGBoost Phase 3"]["rmse"], 2) == 19.70
assert round(resultats_test["XGBoost Phase 3"]["score_nasa"], 1) == 852.9
assert resultats_test["XGBoost Phase 3"]["nb_surestimes"] == 9

lignes = []
for nom_modele in resultats_test:
    lignes.append(resumer_test(nom_modele, resultats_test[nom_modele]))
tableau_test = pd.DataFrame(lignes).set_index("modele")
tableau_test.round(2)

### D.1 Test apparié sur les 100 moteurs test

**Ce qu'on fait** : erreur absolue de chaque moteur test, Random Forest et LightGBM contre XGBoost Phase 3 (mêmes 100 moteurs). **Comment lire** : différence moyenne négative = le modèle se trompe moins ; avec 100 points seulement, il faut un écart net pour conclure.

In [ ]:
erreurs_absolues_phase3 = np.abs(resultats_test["XGBoost Phase 3"]["predictions"] - y_test_vraie)  # |prediction - verite|, par moteur

lignes = []
for nom_modele in resultats_test:
    if nom_modele == "XGBoost Phase 3":
        continue  # la reference ne se compare pas a elle-meme
    erreurs_absolues_modele = np.abs(resultats_test[nom_modele]["predictions"] - y_test_vraie)
    comparaison = comparer_apparie(erreurs_absolues_modele, erreurs_absolues_phase3)
    ligne = {}
    ligne["modele (vs XGBoost Phase 3)"] = nom_modele
    ligne["n_moteurs_test"] = comparaison["n"]
    ligne["difference_moyenne_erreur_absolue"] = comparaison["difference_moyenne"]  # negatif = modele meilleur
    ligne["moteurs_ou_le_modele_est_meilleur"] = int(np.sum(erreurs_absolues_modele < erreurs_absolues_phase3))
    ligne["p_valeur_t"] = comparaison["p_valeur_t"]
    ligne["p_valeur_wilcoxon"] = comparaison["p_valeur_wilcoxon"]
    lignes.append(ligne)
tableau_tests_test = pd.DataFrame(lignes).set_index("modele (vs XGBoost Phase 3)")
tableau_tests_test.round(4)

### D.2 Le score NASA du test, moteur par moteur

**Pourquoi** : le score NASA est une somme d'exponentielles, donc une seule grosse surestimation peut en faire l'essentiel. **Ce qu'on fait** : (1) on compare la pénalité NASA de chaque moteur (test apparié) ; (2) on liste les 5 moteurs qui pèsent le plus dans le score de la Phase 3 ; (3) on recalcule le score total **sans** le moteur le plus pénalisé.

In [ ]:
penalites_test = {}  # nom -> penalite NASA de chacun des 100 moteurs test (un point par moteur)
for nom_modele in resultats_test:
    # metriques_par_moteur renvoie un tableau ; on garde sa colonne "nasa" sous forme de tableau numpy
    penalites_test[nom_modele] = metriques_par_moteur(y_test_vraie, resultats_test[nom_modele]["predictions"],
                                                      moteurs_test_ordre)["nasa"].to_numpy()

# (1) test apparie sur la penalite NASA par moteur
lignes = []
for nom_modele in resultats_test:
    if nom_modele == "XGBoost Phase 3":
        continue
    comparaison = comparer_apparie(penalites_test[nom_modele], penalites_test["XGBoost Phase 3"])
    ligne = {}
    ligne["modele (vs XGBoost Phase 3)"] = nom_modele
    ligne["difference_moyenne_penalite_nasa"] = comparaison["difference_moyenne"]
    ligne["moteurs_ou_le_modele_est_meilleur"] = int(np.sum(penalites_test[nom_modele] < penalites_test["XGBoost Phase 3"]))
    ligne["p_valeur_t"] = comparaison["p_valeur_t"]
    ligne["p_valeur_wilcoxon"] = comparaison["p_valeur_wilcoxon"]
    lignes.append(ligne)
print("Penalite NASA par moteur test, tests apparies contre XGBoost Phase 3 :")
print(pd.DataFrame(lignes).set_index("modele (vs XGBoost Phase 3)").round(4).to_string())

# (2) les 5 moteurs qui pesent le plus dans le score NASA de la Phase 3, et leur penalite avec chaque modele
ordre = np.argsort(-penalites_test["XGBoost Phase 3"])  # positions triees du plus penalise au moins penalise
lignes = []
for position in ordre[:5]:  # les 5 premieres positions
    ligne = {}
    ligne["unit_number"] = moteurs_test_ordre[position]
    ligne["rul_vraie"] = y_test_vraie[position]
    for nom_modele in resultats_test:
        ligne[nom_modele] = penalites_test[nom_modele][position]
    lignes.append(ligne)
print("\nLes 5 moteurs les plus penalises par la Phase 3 (penalite NASA de chaque modele) :")
print(pd.DataFrame(lignes).set_index("unit_number").round(1).to_string())

# (3) score total avec et sans le moteur le plus penalise par la Phase 3
position_pire = ordre[0]
print(f"\nScore NASA test total, avec et sans le moteur {moteurs_test_ordre[position_pire]} :")
for nom_modele in resultats_test:
    total = float(np.sum(penalites_test[nom_modele]))  # le score NASA du test = somme des 100 penalites
    sans_le_pire = total - penalites_test[nom_modele][position_pire]
    print(f"  {nom_modele:40s} : {total:7.1f} avec, {sans_le_pire:7.1f} sans")

### D.3 Figure : prédictions contre vraies RUL du test

**Comment lire** : un panneau par modèle ; la diagonale est la prédiction parfaite, au-dessus = surestimation. Les moteurs surestimés de plus de 20 cycles (au-dessus de la ligne pointillée) sont en triangles orange.

In [ ]:
noms_figure_test = ["Random Forest", "XGBoost Phase 3", "LightGBM"]  # meme ordre que la figure B.3
fig, axes = plt.subplots(1, 3, figsize=(18, 6.5), sharex=True, sharey=True)  # memes axes pour comparer les panneaux

# bord des axes : la plus grande RUL (vraie ou predite) des trois modeles, plus 5 cycles de marge
limite = y_test_vraie.max()
for nom_modele in noms_figure_test:
    limite = max(limite, resultats_test[nom_modele]["predictions"].max())
limite = limite + 5

for ax, nom_modele in zip(axes, noms_figure_test):  # un panneau par modele
    predictions = resultats_test[nom_modele]["predictions"]
    ecarts = predictions - y_test_vraie
    surestime = ecarts > SEUIL_SURESTIMATION  # Vrai pour les moteurs surestimes de plus de 20 cycles

    ax.scatter(y_test_vraie[~surestime], predictions[~surestime], alpha=0.6, color=COULEUR_1, label="moteurs test")  # ~ : "non"
    ax.scatter(y_test_vraie[surestime], predictions[surestime], alpha=0.9, color=COULEUR_2, marker="^", s=60,
               label=f"surestimés > {SEUIL_SURESTIMATION} cycles ({int(surestime.sum())})")
    ax.plot([0, limite], [0, limite], linestyle="--", color="black", label="prédiction = vérité")
    ax.plot([0, limite], [SEUIL_SURESTIMATION, limite + SEUIL_SURESTIMATION], linestyle=":", color=COULEUR_TEXTE,
            label=f"vérité + {SEUIL_SURESTIMATION}")
    resultats = resultats_test[nom_modele]
    ax.set_title(f"{nom_modele}\nRMSE={resultats['rmse']:.2f}, score NASA={resultats['score_nasa']:.1f}, "
                 f"surestimés>20={resultats['nb_surestimes']}/100", fontsize=10)
    ax.set_xlabel("RUL vraie (RUL_FD001.txt)")
    ax.set_xlim(0, limite)
    ax.set_ylim(0, limite)
    ax.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("RUL prédite (modèle entraîné sur les 100 moteurs)")
fig.suptitle("Test officiel FD001 — trois modèles à réglages de base, évalués une seule fois")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "05_predictions_vs_vraies_rul_test_base.png"), dpi=150)
plt.show()

### D.4 Conclusion de la section D

- **Contrôle** : XGBoost Phase 3 redonne exactement RMSE 19,70, score NASA 852,9 et 9 surestimés.
- **Les trois modèles sont proches** : RMSE 19,52 (Random Forest), 19,70 (XGBoost), 20,45 (LightGBM) ; 7 alertes manquées et 1 fausse alerte pour chacun.
- **Aucune différence significative avec XGBoost Phase 3** : erreur absolue par moteur p = 0,97 / 0,76 (Random Forest) et 0,08 / 0,14 (LightGBM) ; pénalité NASA par moteur p = 0,25 / 0,59 et 0,66 / 0,08.
- **Un seul moteur pèse lourd** : le moteur 37 (RUL vraie 21) compte pour 223,6 dans le score NASA de la Phase 3, plus du quart des 852,9. Sans lui : 629,3 (XGBoost), 610,2 (Random Forest), 770,5 (LightGBM).

**Lecture honnête** : le test confirme l'égalité de la section B mais, avec 100 points, il n'a pas la puissance statistique pour départager les modèles.

<a id="partie-e"></a>
## E. Les moteurs surestimés de plus de 20 cycles (XGBoost Phase 3)

**Ce qu'on fait** : on liste les moteurs test que XGBoost Phase 3 surestime de plus de 20 cycles, avec le nombre de cycles **observés** dans `test_FD001.txt` (la longueur de la trajectoire dont dispose le modèle). **Pourquoi** : comprendre les erreurs les plus dangereuses. Cette analyse ne sert à choisir aucun modèle.

In [ ]:
# nombre de cycles observes pour chaque moteur test (trajectoire tronquee), dans l'ordre des moteurs
nb_cycles_par_moteur = contexte_test["test"].groupby("unit_number").size()  # nombre de lignes de chaque moteur
moteurs_test = contexte_test["moteurs_test"]  # numeros de moteur, meme ordre que y_test_vraie
nb_cycles_observes = nb_cycles_par_moteur.loc[moteurs_test].to_numpy()  # remis dans l'ordre de moteurs_test

# un tableau des 100 moteurs test pour XGBoost Phase 3
predictions_phase3 = resultats_test["XGBoost Phase 3"]["predictions"]
tableau_moteurs = pd.DataFrame({
    "unit_number": moteurs_test,
    "rul_vraie": y_test_vraie,
    "rul_predite": predictions_phase3,
    "ecart": predictions_phase3 - y_test_vraie,  # positif = surestimation
    "nb_cycles_observes": nb_cycles_observes,
})
tableau_moteurs["surestime_20"] = tableau_moteurs["ecart"] > SEUIL_SURESTIMATION  # Vrai si surestime de plus de 20 cycles

print(f"=== XGBoost Phase 3 : {int(tableau_moteurs['surestime_20'].sum())} moteurs surestimes de plus de 20 cycles ===")
# seulement les moteurs surestimes, du plus grand ecart au plus petit
surestimes_tries = tableau_moteurs[tableau_moteurs["surestime_20"]].sort_values("ecart", ascending=False)
print(surestimes_tries.round(1).to_string(index=False))

### E.1 Tester des hypothèses plutôt que les affirmer

- **H1** : les moteurs surestimés ont des trajectoires observées **plus courtes** (le modèle a vu peu de cycles, le moteur « a l'air » neuf) ;
- **H2** : les moteurs surestimés ont des RUL vraies **différentes** des autres.

**Notion — test de Mann-Whitney** : il compare deux groupes indépendants (surestimés contre autres) par les rangs, sans supposer de loi normale : « les valeurs d'un groupe ont-elles tendance à être plus grandes ? ». **Limite** : 9 moteurs surestimés seulement ; une p-valeur > 0,05 ne prouve pas l'absence d'effet. En complément, une corrélation de Spearman (ρ, entre −1 et +1 : les deux grandeurs montent-elles ensemble ?) sur les **100 moteurs**, entre l'écart signé et chaque variable.

In [ ]:
groupe_surestimes = tableau_moteurs[tableau_moteurs["surestime_20"]]  # les moteurs surestimes de plus de 20 cycles
groupe_autres = tableau_moteurs[~tableau_moteurs["surestime_20"]]  # tous les autres (~ : "non")

lignes = []
for variable in ["nb_cycles_observes", "rul_vraie"]:  # H1 puis H2
    test_mw = stats.mannwhitneyu(groupe_surestimes[variable], groupe_autres[variable], alternative="two-sided")
    correlation = stats.spearmanr(tableau_moteurs["ecart"], tableau_moteurs[variable])  # sur les 100 moteurs
    ligne = {}
    ligne["variable"] = variable
    ligne["n_surestimes"] = len(groupe_surestimes)
    ligne["n_autres"] = len(groupe_autres)
    ligne["mediane_surestimes"] = groupe_surestimes[variable].median()
    ligne["mediane_autres"] = groupe_autres[variable].median()
    ligne["p_valeur_mann_whitney"] = test_mw.pvalue
    ligne["rho_spearman_ecart_100_moteurs"] = correlation.statistic
    ligne["p_valeur_spearman"] = correlation.pvalue
    lignes.append(ligne)
tableau_hypotheses = pd.DataFrame(lignes).set_index("variable")
tableau_hypotheses.round(4)

### E.2 Figure : écart de prédiction selon la trajectoire observée et selon la RUL vraie

**Comment lire** : un point par moteur test ; au-dessus de la ligne pointillée = surestimé de plus de 20 cycles (triangles orange), sous la ligne noire = sous-estimé.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)  # meme axe vertical (l'ecart) pour les deux panneaux

for ax, variable, titre_x in [(axes[0], "nb_cycles_observes", "nombre de cycles observés dans test_FD001"),
                               (axes[1], "rul_vraie", "RUL vraie (RUL_FD001.txt)")]:  # H1 a gauche, H2 a droite
    ax.scatter(groupe_autres[variable], groupe_autres["ecart"], color=COULEUR_1, marker="o", s=35, alpha=0.7,
               label="autres moteurs")
    ax.scatter(groupe_surestimes[variable], groupe_surestimes["ecart"], color=COULEUR_2, marker="^", s=60, alpha=0.9,
               label=f"surestimés > {SEUIL_SURESTIMATION} cycles")
    ax.axhline(0, color="black", linewidth=1)  # ecart nul : prediction parfaite
    ax.axhline(SEUIL_SURESTIMATION, color=COULEUR_TEXTE, linestyle=":", label=f"surestimation de {SEUIL_SURESTIMATION} cycles")
    ax.set_xlabel(titre_x)
axes[0].set_ylabel("écart = RUL prédite − RUL vraie (cycles) ; > 0 = surestimation")
axes[0].set_title("Écart vs longueur de la trajectoire observée")
axes[1].set_title("Écart vs RUL vraie")
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("Test officiel FD001, XGBoost Phase 3 — d'où viennent les surestimations ?")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "05_moteurs_surestimes.png"), dpi=150)
plt.show()

### E.3 Conclusion de la section E

- **9 moteurs surestimés de plus de 20 cycles**, de +22,6 (moteur 2) à +54,1 cycles (moteur 37).
- **H1 (trajectoires plus courtes) : non démontrée.** Médiane de 121 cycles observés contre 137, mais Mann-Whitney p = 0,07 ; sur les 100 moteurs, ρ = −0,19 (p = 0,06) : au mieux une faible tendance.
- **H2 (RUL vraie différente) : non démontrée entre les deux groupes** (médiane 63 contre 90 ; p = 0,12). La corrélation entre écart et RUL vraie sur 100 moteurs est nette (ρ = −0,34, p < 0,001) mais en partie **mécanique** : la cible apprise est plafonnée à 125, donc un moteur dont la RUL vraie dépasse largement 125 est presque forcément sous-estimé.
- **Les plus dangereux** : les moteurs 37, 18 et 41 (RUL vraie 21, 28 et 18) sont prédits à 75, 62 et 50 cycles, au-dessus du seuil de 30 : ce sont des alertes manquées, sur des trajectoires de longueur ordinaire (121 à 133 cycles). **Piste non testée** : examiner leurs capteurs.

<a id="partie-f"></a>
## F. Synthèse et fichiers de résultats

### F.1 Tableau final

**Comment lire** : pour chaque modèle évalué sur le test, les colonnes de validation croisée (mêmes 5 plis) puis celles du test officiel (évalué une fois).

In [ ]:
lignes = []
for nom_modele in resultats_test:  # meme ordre que le tableau de la section D
    ligne_cv = resumer_cv(nom_modele, resultats_cv_base[nom_modele], NB_MOTEURS_TRAIN)  # validation croisee (section B)
    ligne_test = resumer_test(nom_modele, resultats_test[nom_modele])  # test officiel (section D)
    ligne = {}
    ligne["modele"] = nom_modele
    ligne["rmse_cv"] = ligne_cv["rmse_cv_moyen"]
    ligne["nasa_cv"] = ligne_cv["nasa_cv_moyen"]
    ligne["surestimes_oof"] = ligne_cv["surestimes_oof"]
    ligne["rmse_test"] = ligne_test["rmse_test"]
    ligne["rmse_test_plafonnee"] = ligne_test["rmse_test_rul_plafonnee"]
    ligne["nasa_test"] = ligne_test["nasa_test"]
    ligne["surestimes_test"] = ligne_test["surestimes_test"]
    ligne["alertes_manquees_test"] = ligne_test["alertes_manquees_test"]
    ligne["fausses_alertes_test"] = ligne_test["fausses_alertes_test"]
    lignes.append(ligne)
tableau_synthese = pd.DataFrame(lignes).set_index("modele")
tableau_synthese.round(2)

### F.2 Écriture des résultats

**Ce qu'on fait** : on écrit deux fichiers dans `docs/resultats/` : `05_comparaison_cv.csv` (les modèles des sections B et C, une ligne par modèle) et `05_comparaison_test.csv` (les trois modèles évalués sur le test, section D).

In [ ]:
# 1) validation croisee : sections B et C, une ligne par modele
lignes = []
for nom_modele in resultats_cv_base:
    ligne = resumer_cv(nom_modele, resultats_cv_base[nom_modele], NB_MOTEURS_TRAIN)
    ligne["section"] = "B - reglages de base"
    lignes.append(ligne)
for reglage in reglages_penalite:
    ligne = resumer_cv(reglage["reglage"], resultats_cv_penalite[reglage["reglage"]], NB_MOTEURS_TRAIN)
    ligne["section"] = "C - penalite de surestimation"
    lignes.append(ligne)
tableau_comparaison_cv = pd.DataFrame(lignes)
chemin_cv = os.path.join(DOSSIER_RESULTATS, "05_comparaison_cv.csv")
tableau_comparaison_cv.to_csv(chemin_cv, index=False)  # index=False : pas de colonne de numeros de ligne

# 2) test officiel : section D
tableau_comparaison_test = tableau_test.reset_index()  # "modele" redevient une colonne ordinaire
chemin_test = os.path.join(DOSSIER_RESULTATS, "05_comparaison_test.csv")
tableau_comparaison_test.to_csv(chemin_test, index=False)

print("Fichiers ecrits :")
print(" ", os.path.relpath(chemin_cv))
print(" ", os.path.relpath(chemin_test))

### F.3 Ce qui est établi, ce qui ne l'est pas

**Établi** (validation croisée, mêmes 5 plis, tests appariés sur 100 moteurs)
- Les trois ensembles d'arbres battent largement la référence naïve et sont **à égalité entre eux** à réglages de base : XGBoost se justifie par sa vitesse et sa perte sur mesure, pas par sa précision.
- Changer les constantes 13/10 du score ne change aucune prédiction. Pénaliser la surestimation **dans l'apprentissage** la réduit, avec un coût en RMSE, en fausses alertes et en biais qui augmente avec α.
- Les chiffres de la Phase 3 sont retrouvés à l'identique (contrôles des sections B et D).

**Non concluant**
- Sur le test (100 moteurs), aucune différence significative avec XGBoost Phase 3 ; les alertes manquées restent à 7 pour chacun des trois modèles. Les hypothèses H1 et H2 sur les moteurs surestimés : trop peu de moteurs pour conclure.

**Limites et suite**
- Ce notebook ne traite que FD001 et les modèles à arbres : le LSTM et le CNN 1D sont dans `notebooks/03b_deep_learning_lstm_cnn.ipynb`, FD002 à FD004 dans `notebooks/01b_sous_jeux_fd002_fd004.ipynb`.
- La grille de la section C est une expérience à réglages fixes : aucun modèle « plus prudent » n'est retenu ici.
- Seuils d'alerte (30 cycles) et de surestimation (20 cycles) : hypothèses de travail, à discuter.
- La validation croisée note tous les cycles, le test seulement le dernier cycle d'une trajectoire tronquée : une validation qui imite cette troncature serait plus fidèle. Les capteurs des moteurs 37, 18 et 41 restent à examiner.

### F.4 Contrôle des chiffres de l'encadré « En 30 secondes »

**Ce qu'on fait** : on relit les deux fichiers qui viennent d'être écrits et on vérifie que les chiffres cités dans l'encadré du début sont bien ceux-là. **Pourquoi** : si un résultat change un jour, cette cellule s'arrête sur une erreur au lieu de laisser un texte faux en tête du notebook.

In [ ]:
# relecture des deux fichiers ecrits en F.2 ; la colonne "modele" sert d'etiquette de ligne
cv_relu = pd.read_csv(chemin_cv).set_index("modele")
test_relu = pd.read_csv(chemin_test).set_index("modele")

# RMSE CV des trois ensembles d'arbres (section B)
for nom_modele in ["Random Forest", "XGBoost Phase 3", "LightGBM"]:
    print(f"RMSE CV {nom_modele:16s} : {cv_relu.loc[nom_modele, 'rmse_cv_moyen']:.2f}")
p_valeur_min = tableau_tests_base.loc[["Random Forest", "LightGBM"], ["p_valeur_t", "p_valeur_wilcoxon"]].min().min()
print(f"Plus petite p-valeur des tests apparies contre XGBoost (section B.2) : {p_valeur_min:.4f}")

# grille de la section C : XGBoost, perte quadratique asymetrique, alpha = 1 puis alpha = 3
alpha_1 = cv_relu.loc["XGBoost quadratique alpha=1"]
alpha_3 = cv_relu.loc["XGBoost quadratique alpha=3"]
baisse_surestimes = 100.0 * (1.0 - alpha_3["surestimes_oof"] / alpha_1["surestimes_oof"])  # en pourcentage
hausse_fausses_alertes = 100.0 * (alpha_3["fausses_alertes_oof"] / alpha_1["fausses_alertes_oof"] - 1.0)
print(f"alpha 1 -> 3 : surestimes OOF > 20 {int(alpha_1['surestimes_oof'])} -> {int(alpha_3['surestimes_oof'])} "
      f"(-{baisse_surestimes:.0f} %), RMSE CV {alpha_1['rmse_cv_moyen']:.2f} -> {alpha_3['rmse_cv_moyen']:.2f}, "
      f"fausses alertes +{hausse_fausses_alertes:.0f} %")

# test officiel de XGBoost Phase 3 (section D)
test_phase3 = test_relu.loc["XGBoost Phase 3"]
print(f"Test officiel XGBoost Phase 3 : RMSE {test_phase3['rmse_test']:.2f}, score NASA {test_phase3['nasa_test']:.1f}")

# les valeurs ecrites dans l'encadre (arrondies comme dans le texte)
assert round(cv_relu.loc["Random Forest", "rmse_cv_moyen"], 2) == 16.96
assert round(cv_relu.loc["XGBoost Phase 3", "rmse_cv_moyen"], 2) == 16.87
assert round(cv_relu.loc["LightGBM", "rmse_cv_moyen"], 2) == 16.85
assert p_valeur_min > 0.5
assert int(alpha_1["surestimes_oof"]) == 2386 and int(alpha_3["surestimes_oof"]) == 1590
assert round(baisse_surestimes) == 33
assert round(alpha_1["rmse_cv_moyen"], 2) == 16.86 and round(alpha_3["rmse_cv_moyen"], 2) == 17.68
assert round(hausse_fausses_alertes) == 64
assert round(test_phase3["rmse_test"], 2) == 19.70 and round(test_phase3["nasa_test"], 1) == 852.9
print("Les chiffres de l'encadre sont bien ceux des fichiers de resultats.")

**Durée** : temps de calcul des sections A à F (la section G, interactive, est chronométrée à part).

In [ ]:
duree_notebook = time.perf_counter() - debut_notebook  # secondes ecoulees depuis la cellule A.2
print(f"Sections A a F executees en {duree_notebook / 60:.1f} minutes.")

<a id="partie-g"></a>
## G. Démo : tester un moteur que les modèles n'ont jamais vu

**Le principe** : les modèles ont appris uniquement sur `train_FD00X.txt`. On tire au hasard un moteur de `test_FD00X.txt` ; à chaque vol (un cycle), le modèle ne reçoit que les mesures jusqu'à ce vol, comme en service. La vraie RUL, lue dans le fichier `RUL_FD00X.txt` de la NASA, n'est montrée qu'après.
**Comment** : `src/inference.py` fait tout le calcul en direct (préparation apprise sur le train, puis prédiction) ; aucun fichier de prédictions précalculées n'est lu.
**Astuce** : cette section peut s'exécuter seule, après un redémarrage du noyau : sa première cellule refait les imports nécessaires.

### G.1 Réglages de la démo

**À modifier avant de lancer** : `SOUS_JEU` (FD001 à FD004) ; `GRAINE` (`None` = un moteur différent à chaque exécution de G.3, un nombre = toujours le même moteur, pour rejouer une démo) ; la liste des modèles comparés : XGBoost Phase 3, LSTM et CNN 1D.

In [ ]:
import os
import random
import sys
import time

# racine du depot importable (utile si cette section est lancee seule, apres un redemarrage du noyau)
RACINE_DEPOT_DEMO = os.path.abspath("..")  # le notebook est dans notebooks/ : la racine est le dossier parent
if RACINE_DEPOT_DEMO not in sys.path:  # on ne l'ajoute qu'une fois
    sys.path.insert(0, RACINE_DEPOT_DEMO)

import matplotlib.pyplot as plt

# ORDRE VOULU : src.inference importe xgboost AVANT torch, puis regle torch sur 1 coeur (voir src/deep_learning.py) ;
# importer torch soi-meme avant xgboost peut bloquer ou faire planter le noyau
from src import inference

SOUS_JEU = "FD001"  # sous-jeu NASA : "FD001", "FD002", "FD003" ou "FD004"
GRAINE = None       # None : un moteur different a chaque execution de G.3 ; un entier (ex. 7) : toujours le meme moteur
MODELES_DEMO = ["xgboost_phase3", "lstm", "cnn1d"]  # cles des modeles dans src/inference.py
SEUIL_ALERTE_DEMO = 30  # cycles : RUL predite <= 30 -> alerte maintenance (hypothese de travail, comme en A.2)
COULEURS_DEMO = ["#2a78d6", "#eb6834", "#1baf7a"]  # bleu, orange, vert d'eau (memes teintes qu'en A.2)
debut_demo = time.perf_counter()  # chronometre de la section G

### G.2 Chargement des modèles

**Ce qu'on fait** : `charger_modeles` apprend la préparation sur `train_<SOUS_JEU>` (capteurs retirés, min / max des capteurs, régimes de vol), relit XGBoost Phase 3 dans `models/` (ou le réentraîne en mémoire s'il manque ou ne correspond plus à la préparation, sans rien écrire sur le disque) et relit les poids du LSTM et du CNN 1D (jamais réentraînés ici). **Comment lire** : un modèle indisponible (fichier absent) est signalé avec la raison, les autres fonctionnent.

In [ ]:
# sauvegarder=False : si XGBoost doit etre reentraine, il reste en memoire (le notebook n'ecrit rien dans models/)
inference_demo = inference.charger_modeles(SOUS_JEU, modeles=MODELES_DEMO, sauvegarder=False)

print(f"Sous-jeu {SOUS_JEU} : chargement en {inference_demo['duree_chargement_s']:.1f} s")
for modele in inference_demo["modeles"]:  # les modeles prets a predire
    print(f"  disponible   : {inference.nom_modele(modele):16s} ({inference_demo['modeles'][modele]['description']})")
for modele in inference_demo["indisponibles"]:  # par exemple un fichier de poids absent
    print(f"  indisponible : {inference.nom_modele(modele):16s} -> {inference_demo['indisponibles'][modele]}")
for message in inference_demo["messages"]:  # par exemple : XGBoost reentraine en memoire faute de fichier
    print("  note :", message)

### G.3 Prédiction en direct sur un moteur mystère

**Ce qu'on fait** : on tire un moteur de test au hasard sans montrer son numéro. (1) Au dernier vol observé n, chaque modèle reçoit les mesures des vols 1 à n et prédit la RUL, avec son temps de calcul. (2) On refait la prédiction à chaque vol t, avec les seules mesures 1 à t, pour tracer une courbe.
**Comment lire** : une courbe par modèle ; sous la ligne pointillée (30 cycles), le modèle déclencherait une alerte maintenance. Relancer cette cellule tire un nouveau moteur (si `GRAINE = None`).

In [ ]:
def tracer_courbes_demo(trajectoire, modeles, titre, verite=None):
    """
    Trace la RUL predite a chaque vol par chaque modele, et la vraie RUL si elle est fournie.

    Parameters
    ----------
    trajectoire : pandas.DataFrame
        Le resultat de `inference.predire_trajectoire` : "time_cycles" et une colonne par modele.
    modeles : list de str
        Les cles des modeles a tracer (colonnes de `trajectoire`).
    titre : str
        Le titre de la figure.
    verite : pandas.DataFrame ou None, optionnel
        Le resultat de `inference.verite_moteur` ("time_cycles", "rul_vraie", "rul_vraie_cappee") ;
        None (defaut) : la verite n'est pas dessinee.

    Returns
    -------
    None
    """
    fig, ax = plt.subplots(figsize=(13, 5))
    for numero, modele in enumerate(modeles):  # une courbe et une couleur par modele
        couleur = COULEURS_DEMO[numero % len(COULEURS_DEMO)]  # % : on reprend la liste au debut s'il manque des couleurs
        ax.plot(trajectoire["time_cycles"], trajectoire[modele], color=couleur, linewidth=2,
                label=f"RUL prédite — {inference.nom_modele(modele)}")
    if verite is not None:  # la verite n'est dessinee qu'une fois revelee (cellule G.4)
        ax.plot(verite["time_cycles"], verite["rul_vraie"], color="black", linewidth=2,
                label=f"vraie RUL (RUL_{SOUS_JEU}.txt)")
        ax.plot(verite["time_cycles"], verite["rul_vraie_cappee"], color="black", linestyle="--", linewidth=1,
                label="vraie RUL plafonnée à 125 (la cible apprise)")
    ax.axhline(SEUIL_ALERTE_DEMO, color="#52514e", linestyle=":", label=f"seuil d'alerte ({SEUIL_ALERTE_DEMO} cycles)")
    ax.set_ylim(bottom=0)  # l'axe vertical part de 0 cycle restant
    ax.set_xlabel("vol (cycle) observé")
    ax.set_ylabel("RUL (cycles restants)")
    ax.set_title(titre)
    ax.legend(fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1.0))  # legende a droite, hors des courbes
    plt.tight_layout()
    plt.show()


# 1) tirage du moteur mystere
if GRAINE is None:
    # alea fourni par le systeme d'exploitation : jamais influence par une graine fixee ailleurs dans le notebook
    graine_tirage = random.SystemRandom().randrange(1_000_000)  # entier entre 0 et 999 999
else:
    graine_tirage = GRAINE
moteur_mystere = inference.tirer_moteur_test(SOUS_JEU, graine=graine_tirage)  # meme graine = meme moteur
mesures_mystere = moteur_mystere["mesures"]  # mesures brutes des vols 1 a n (colonnes de test_<SOUS_JEU>.txt)
nb_vols = moteur_mystere["nb_cycles"]  # n : nombre de vols observes

print(f"Moteur mystere de test_{SOUS_JEU}.txt : numero masque, {nb_vols} vols observes")
print(f"Graine du tirage : {graine_tirage} (mettre GRAINE = {graine_tirage} en G.1 pour rejouer ce moteur)")

# 2) prediction au dernier vol observe : chaque modele recoit les mesures des vols 1 a n, rien d'autre
au_dernier_vol = inference.predire_au_cycle(inference_demo, mesures_mystere, nb_vols)
print(f"\nRUL predite au vol {nb_vols} (preparation des mesures : {au_dernier_vol['temps_preparation_ms']:.1f} ms) :")
for modele in au_dernier_vol["predictions"]:  # un modele a la fois
    print(f"  {inference.nom_modele(modele):16s} : {au_dernier_vol['predictions'][modele]:6.1f} cycles "
          f"(calcul : {au_dernier_vol['temps_ms'][modele]:.1f} ms)")

# 3) prediction a chaque vol t, avec les seules mesures 1 a t (une troncature et une preparation par vol)
debut = time.perf_counter()
trajectoire_mystere = inference.predire_trajectoire(inference_demo, mesures_mystere)  # une ligne par vol
duree_trajectoire = time.perf_counter() - debut
modeles_traces = list(au_dernier_vol["predictions"].keys())  # les modeles disponibles, dans l'ordre d'affichage
print(f"\nCourbe : {nb_vols} vols x {len(modeles_traces)} modeles recalcules en {duree_trajectoire:.1f} s "
      f"(environ {1000 * duree_trajectoire / nb_vols:.0f} ms par vol)")

tracer_courbes_demo(trajectoire_mystere, modeles_traces, f"Moteur mystère ({SOUS_JEU}) : RUL prédite à chaque vol")

### G.4 Révéler la vérité

**Ce qu'on fait** : on affiche le numéro du moteur, sa vraie RUL au dernier vol (`RUL_<SOUS_JEU>.txt`), l'écart de chaque modèle (positif = surestimation, le cas dangereux) et son statut d'alerte, puis la courbe avec la vérité.
**Règle d'alerte** (la même que `classer_alerte` de `src/alertes.py` et que les colonnes d'alertes des sections B à D) : le modèle alerte si sa RUL prédite ≤ 30 ; il aurait dû alerter si la vraie RUL ≤ 30. Les modèles ont appris une cible plafonnée à 125 : si la vraie RUL dépasse 125, une prédiction plus basse (écart négatif) est attendue.

In [ ]:
def statut_alerte(rul_predite, rul_vraie, seuil):
    """
    Classe une prediction en quatre cas, avec la meme regle que `classer_alerte` de src/alertes.py.

    Le modele declenche une alerte si la RUL predite est <= seuil ; il aurait du alerter si la vraie
    RUL est <= seuil.

    Parameters
    ----------
    rul_predite : float
        La RUL predite (cycles).
    rul_vraie : float
        La vraie RUL (cycles).
    seuil : float
        Le seuil d'alerte (cycles).

    Returns
    -------
    str
        "alerte correcte", "ALERTE MANQUEE" (le cas dangereux), "fausse alerte" ou "rien a signaler".
    """
    alerte_declenchee = rul_predite <= seuil  # decision du modele
    aurait_du_alerter = rul_vraie <= seuil  # ce qu'il aurait fallu faire
    if alerte_declenchee and aurait_du_alerter:
        return "alerte correcte"
    if aurait_du_alerter:  # il fallait alerter et le modele ne l'a pas fait
        return "ALERTE MANQUEE"
    if alerte_declenchee:  # alerte alors que le moteur avait encore de la marge (maintenance inutile)
        return "fausse alerte"
    return "rien a signaler"


rul_vraie_finale = moteur_mystere["rul_finale"]  # ligne de ce moteur dans RUL_<SOUS_JEU>.txt
print(f"Le moteur mystere etait le moteur n°{moteur_mystere['moteur']} de test_{SOUS_JEU}.txt")
print(f"Vraie RUL au vol {nb_vols} : {rul_vraie_finale:.0f} cycles (RUL_{SOUS_JEU}.txt)\n")
for modele in au_dernier_vol["predictions"]:
    rul_predite = au_dernier_vol["predictions"][modele]
    ecart = rul_predite - rul_vraie_finale  # positif = surestimation
    statut = statut_alerte(rul_predite, rul_vraie_finale, SEUIL_ALERTE_DEMO)
    print(f"  {inference.nom_modele(modele):16s} : predit {rul_predite:6.1f} cycles, ecart {ecart:+6.1f} -> {statut}")

verite_mystere = inference.verite_moteur(mesures_mystere, rul_vraie_finale)  # vraie RUL a chaque vol
tracer_courbes_demo(trajectoire_mystere, modeles_traces,
                    f"Moteur n°{moteur_mystere['moteur']} ({SOUS_JEU}) : prédictions et vérité", verite=verite_mystere)

### G.5 Lancer la démo interactive (application Streamlit)

**Ce qu'on fait** : on démarre `app/tableau_de_bord.py` en arrière-plan avec le Python du projet (`.venv`), depuis la racine du dépôt, sur le premier port libre à partir de 8599 : le notebook reste utilisable pendant que l'application tourne.
**Comment** : `subprocess.Popen` lance l'application sans attendre sa fin ; on interroge son adresse `/_stcore/health` jusqu'à la réponse « ok », puis on affiche le lien à ouvrir dans le navigateur.
**Attention** : « Exécuter tout » démarre puis arrête aussitôt l'application (dernière cellule). Pour la présentation, relancer cette cellule seule.

In [ ]:
import atexit          # pour arreter l'application si Python se ferme sans passer par la cellule G.6
import socket          # pour tester si un port est libre
import subprocess      # pour lancer un autre programme (l'application) sans bloquer le notebook
import tempfile        # dossier temporaire du systeme, pour le journal de l'application
import urllib.request  # pour interroger l'application par HTTP

from IPython.display import Markdown, display  # pour afficher un lien cliquable

PORT_DE_DEPART = 8599  # premier port essaye ; s'il est deja pris (par une autre application), on essaie le suivant
DELAI_DEMARRAGE_MAX_S = 60  # temps maximal d'attente du demarrage, en secondes


def port_libre(port):
    """
    Dit si un port de cet ordinateur est libre, en essayant de le reserver un court instant.

    Parameters
    ----------
    port : int
        Le numero de port.

    Returns
    -------
    bool
        Vrai si aucun programme n'utilise ce port.
    """
    essai = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        essai.bind(("127.0.0.1", port))  # echoue si un autre programme utilise deja ce port
        libre = True
    except OSError:
        libre = False
    essai.close()  # on libere le port tout de suite
    return libre


def premier_port_libre(port_depart, nb_essais=50):
    """
    Renvoie le premier port libre a partir de port_depart.

    Parameters
    ----------
    port_depart : int
        Le premier port essaye.
    nb_essais : int, optionnel
        Nombre de ports essayes au maximum (50 par defaut).

    Returns
    -------
    int
        Le premier port libre.
    """
    for port in range(port_depart, port_depart + nb_essais):  # 8599, 8600, 8601, ...
        if port_libre(port):
            return port
    raise RuntimeError(f"Aucun port libre entre {port_depart} et {port_depart + nb_essais - 1}.")


def arreter_processus(processus):
    """
    Arrete l'application : demande d'arret, puis arret force si elle ne s'est pas arretee en 10 secondes.

    Parameters
    ----------
    processus : subprocess.Popen
        Le processus lance par `subprocess.Popen`.

    Returns
    -------
    str
        "deja arretee", "arretee" ou "arretee de force".
    """
    if processus.poll() is not None:  # poll() renvoie None tant que le processus tourne
        return "deja arretee"
    processus.terminate()  # demande d'arret (signal SIGTERM)
    try:
        processus.wait(timeout=10)  # on attend la fin, au plus 10 secondes
        return "arretee"
    except subprocess.TimeoutExpired:
        processus.kill()  # arret force (signal SIGKILL)
        processus.wait()
        return "arretee de force"


def attendre_application(processus, url_sante, delai_max_s):
    """
    Interroge l'adresse de sante de l'application jusqu'a la reponse "ok".

    Parameters
    ----------
    processus : subprocess.Popen
        Le processus de l'application (pour detecter un arret premature).
    url_sante : str
        L'adresse a interroger (".../_stcore/health").
    delai_max_s : float
        Temps maximal d'attente, en secondes.

    Returns
    -------
    bool
        Vrai si l'application a repondu "ok" a temps.
    """
    debut = time.perf_counter()
    while time.perf_counter() - debut < delai_max_s:
        if processus.poll() is not None:  # l'application s'est deja arretee : erreur au demarrage
            return False
        try:
            with urllib.request.urlopen(url_sante, timeout=2) as reponse:
                if reponse.read().decode().strip() == "ok":
                    return True
        except OSError:  # pas encore prete (connexion refusee) : on reessaie un peu plus tard
            pass
        time.sleep(0.5)  # demi-seconde entre deux essais
    return False


# python du projet : .venv/bin/python s'il existe (macOS, Linux), sinon le python de ce noyau
python_demo = os.path.join(RACINE_DEPOT_DEMO, ".venv", "bin", "python")
if not os.path.exists(python_demo):
    python_demo = sys.executable

port_demo = premier_port_libre(PORT_DE_DEPART)
url_demo = f"http://127.0.0.1:{port_demo}"
chemin_journal = os.path.join(tempfile.gettempdir(), f"aero_predict_streamlit_{port_demo}.log")  # messages de l'application
journal = open(chemin_journal, "w", encoding="utf-8")

commande = [python_demo, "-m", "streamlit", "run", "app/tableau_de_bord.py",
            "--server.port", str(port_demo),
            "--server.address", "127.0.0.1",        # accessible depuis cet ordinateur seulement
            "--server.headless", "true",            # n'ouvre pas de navigateur tout seul
            "--browser.gatherUsageStats", "false"]  # aucune statistique d'usage envoyee
# Popen lance la commande et rend la main tout de suite ; cwd : depuis la racine du depot
processus_demo = subprocess.Popen(commande, cwd=RACINE_DEPOT_DEMO, stdout=journal, stderr=subprocess.STDOUT)
atexit.register(arreter_processus, processus_demo)  # filet de securite : arret quand le noyau Python se ferme

if attendre_application(processus_demo, f"{url_demo}/_stcore/health", DELAI_DEMARRAGE_MAX_S):
    print(f"Application demarree (processus {processus_demo.pid}, port {port_demo}, journal : {chemin_journal}).")
    display(Markdown(f"**Démo prête** : [{url_demo}]({url_demo}) — à ouvrir dans le navigateur."))
else:
    arreter_processus(processus_demo)
    journal.close()
    with open(chemin_journal, encoding="utf-8") as fichier:
        print(fichier.read()[-2000:])  # les 2 000 derniers caracteres du journal, pour comprendre l'erreur
    raise RuntimeError(f"L'application n'a pas repondu en {DELAI_DEMARRAGE_MAX_S} s (journal ci-dessus).")

### G.6 Arrêter la démo

**Ce qu'on fait** : on arrête l'application lancée en G.5 (demande d'arrêt, puis arrêt forcé si elle ne répond pas en 10 secondes) et on ferme son journal. À lancer à la fin de la présentation ; c'est aussi la dernière cellule du notebook, pour qu'une exécution complète ne laisse aucun programme tourner.

In [ ]:
etat = arreter_processus(processus_demo)  # sans effet si l'application est deja arretee
journal.close()  # ferme le fichier des messages de l'application
print(f"Application du port {port_demo} : {etat}.")
print(f"Section G executee en {time.perf_counter() - debut_demo:.0f} s.")